In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# --- تنظیمات آدرس‌ها ---
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output2.xlsx'

# سنسورهایی که به عنوان فیچر استفاده می‌شوند
target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

def run_dbscan_analysis():
    if not os.path.exists(file_path):
        print(f"❌ فایل یافت نشد: {file_path}")
        return

    # ۱. خواندن داده‌ها
    df = pd.read_excel(file_path)
    
    # بررسی موجود بودن ستون‌ها
    available_sensors = [col for col in target_sensors if col in df.columns]
    
    if not available_sensors:
        print("❌ سنسورهای هدف یافت نشدند.")
        return

    # ۲. آماده‌سازی و استانداردسازی (DBSCAN به مقیاس داده‌ها بسیار حساس است)
    df_model = df.dropna(subset=available_sensors).copy()
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_model[available_sensors])

    # ۳. اجرای DBSCAN
    # eps: شعاع همسایگی - اگر تعداد آنومالی‌ها خیلی زیاد شد، این عدد را کمی بزرگتر کنید (مثلاً 0.7)
    # min_samples: حداقل نقاط برای تشکیل یک خوشه
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    clusters = dbscan.fit_predict(scaled_data)
    
    df_model['Cluster_Labels'] = clusters

    # ۴. تحلیل خروجی DBSCAN
    # در این الگوریتم، مقدار -1 یعنی نقطه به عنوان نویز (Outlier) شناسایی شده است
    df_model['Is_Anomaly'] = df_model['Cluster_Labels'].apply(lambda x: 'Yes' if x == -1 else 'No')
    
    # برچسب‌گذاری وضعیت‌ها
    def label_states(row):
        if row['Cluster_Labels'] == -1:
            return 'Noise/Anomaly'
        else:
            return f'Cluster_{row["Cluster_Labels"]}'

    df_model['System_State_Label'] = df_model.apply(label_states, axis=1)

    # ۵. ذخیره در اکسل
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        df_model.to_excel(output_filename, index=False)
        
        # گزارش کوتاه در کنسول
        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
        n_noise = list(clusters).count(-1)
        
        print(f"🚀 تحلیل DBSCAN با موفقیت انجام شد!")
        print(f"✅ تعداد خوشه‌های شناسایی شده: {n_clusters}")
        print(f"⚠️ تعداد نقاط نویز (آنومالی): {n_noise}")
        print(f"📁 فایل خروجی: {output_filename}")
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")

if __name__ == "__main__":
    run_dbscan_analysis()

🚀 تحلیل DBSCAN با موفقیت انجام شد!
✅ تعداد خوشه‌های شناسایی شده: 9
⚠️ تعداد نقاط نویز (آنومالی): 56
📁 فایل خروجی: outputs\G11\dsas_g11_bearings_vibration_temp_clustering\clustering\dsas_g11_clustering_output2.xlsx
